In [1]:
import os

In [2]:
%pwd

'c:\\Users\\sagal\\Desktop\\Let us build\\emotion_detection\\research\\03_inference_trials'

In [3]:
os.chdir('../../')

In [4]:
%pwd

'c:\\Users\\sagal\\Desktop\\Let us build\\emotion_detection'

In [5]:
import pandas as pd
import torch
from torchvision import models
from torchvision.models import resnet18
import torch.nn as nn
from PIL import Image
from torchvision import transforms
import torch.nn.functional as F
import os
import cv2
from PIL import Image
from pathlib import Path
from emotion_detection.config.configuration import configurationManager


## Step 1: Load model

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [7]:
# Load the initial model
NUM_CLASSES = 7

model = resnet18(weights=None)
model.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(model.fc.in_features, 7))

In [8]:
#first import config manager to read config.yaml
config_manager = configurationManager()
config = config_manager.get_model_evaluation_config()

MODEL_PATH = Path(config.model_path)

[2026-08-04 18:15:12,591: INFO: common: YAML file loaded successfully from: config\config.yaml]
[2026-08-04 18:15:12,598: INFO: common: YAML file loaded successfully from: params.yaml]
[2026-08-04 18:15:12,601: INFO: common: created directory at artifacts]
[2026-08-04 18:15:12,605: INFO: common: created directory at artifacts/model_evaluation]


In [9]:
#fix the model path from local device based to config based
MODEL_PATH = Path(config.model_path)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## Step 2 : inference preprocessing

In [10]:
# Introduce inference tansform, same as during training
inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [11]:
# Get test image
#image = Image.open(r"S:\Projects\emotion_detection\artifacts\data_ingestion\Organized\train\happy\train_00036_aligned.jpg")


#local hardcoding and the image name can change so, the fix here would be to choose the best available image dynamically
#instead of hardcoding and we can use pathlib as we have used it earlier

In [12]:
#automatically pick the first image inside the happy folder
happy_folder = Path("artifacts/data_ingestion/Organized/train/happy")
test_image_path = next(happy_folder.glob("*.jpg"))

image = Image.open(test_image_path)
print(f"Loaded image: {test_image_path}")

Loaded image: artifacts\data_ingestion\Organized\train\happy\train_09748_aligned.jpg


In [13]:
# apply transform
image_tensor = inference_transform(image)

In [14]:
print(image_tensor.shape)

torch.Size([3, 224, 224])


In [15]:
# add batch
image_tensor = image_tensor.unsqueeze(0)

In [16]:
print(image_tensor.shape)

torch.Size([1, 3, 224, 224])


In [17]:
image_tensor = image_tensor.to(device)

## Step 3: single image prediction

In [18]:
with torch.no_grad():
    outputs = model(image_tensor)

print(outputs.shape)    

torch.Size([1, 7])


In [19]:
# Convert logits into probabilities
probabilities = F.softmax(outputs, dim=1)
print(probabilities)

tensor([[0.0989, 0.0564, 0.0038, 0.5575, 0.0449, 0.1649, 0.0736]],
       device='cuda:0')


In [20]:
# Get highest probability
confidence, predicted = torch.max(probabilities, dim=1)

In [21]:
# Updated Label map matching RAF-DB training index order
class_names = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

In [22]:
emotion = class_names[predicted.item()]

print(f"Prediction : {emotion}")
print(f"Confidence : {confidence.item()*100:.2f}%")

Prediction : happy
Confidence : 55.75%


In [ ]:
# helper function for testing
#class_names = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


#def predict_emotion(
#    model,
 #   transform,
  #  device,
 #   image_path=None,
 #   frame=None
#):
 #   """
 #   Predict emotion from a single image.
#
  #  Args:
    #    image_path (str): Path to image.
   #     model: Trained PyTorch model.
     #   transform: Inference transform.
      #  device: cuda or cpu.

    #Returns:
     #   predicted_label, confidence
    #"""

    # Load image
   # if image_path is not None:

    #    image = Image.open(image_path).convert("RGB")

   # elif frame is not None:

    #    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
     #   image = Image.fromarray(image)

    #else:
     #   raise ValueError("Provide either image_path or frame.")

    # Preprocess
    #image = transform(image)

    # Add batch dimension
    #image = image.unsqueeze(0).to(device)

    # Prediction
    #model.eval()
    #with torch.no_grad():
     #   outputs = model(image)

      #  probabilities = F.softmax(outputs, dim=1)

       # confidence, predicted = torch.max(probabilities, dim=1)

    #predicted_label = class_names[predicted.item()]
    #confidence = confidence.item() * 100

    # Convert probabilities into a dictionary
    #all_probabilities = {
     #   label: prob * 100
      #  for label, prob in zip(
       #     class_names,
       #     probabilities.squeeze().cpu().numpy()
       # )
   # }

    #return predicted_label, confidence, all_probabilities

In [35]:
class_names = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

import cv2
import torch
import numpy as np
from PIL import Image
import torch.nn.functional as F

def predict_emotion(
    model,
    transform,
    device,
    image_path=None,
    frame=None
):
    """
    Predict emotion from a single image or video frame with automatic face cropping.
    """
    if image_path is not None:
        pil_img = Image.open(image_path).convert("RGB")
        cv_img = cv2.imread(image_path)
    elif frame is not None:
        cv_img = frame
        pil_img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    else:
        raise ValueError("Either image_path or frame must be provided.")

    # Face detection
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    gray = cv2.cvtColor(cv_img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    # Crop face if detected
    if len(faces) > 0:
        x, y, w, h = max(faces, key=lambda rect: rect[2] * rect[3])
        pil_img = pil_img.crop((x, y, x + w, y + h))

    # Preprocessing
    image_tensor = transform(pil_img).unsqueeze(0).to(device)

    # Model inference
    model.eval()
    with torch.no_grad():
        outputs = model(image_tensor)
        probabilities = F.softmax(outputs, dim=1)[0]
        confidence, predicted_idx = torch.max(probabilities, dim=0)

    predicted_label = class_names[predicted_idx.item()]
    confidence_val = confidence.item() * 100

    # Format dictionary mapping class names to percentage probabilities
    probs_dict = {
        label: prob.item() * 100 
        for label, prob in zip(class_names, probabilities)
    }

    return predicted_label, confidence_val, probs_dict

In [36]:
#we are going to do the same here as well, remove hardcoding, use referencing and use the first image available from the directory

In [37]:

#define directry from where you want the image
sad_folder = Path("artifacts/data_ingestion/Organized/train/sad")

#first available image dynamically
test_image_path = next(sad_folder.glob("*.jpg"))

print(f"Testing with image: {test_image_path}")

#pass the path to predict_emotion
prediction, confidence, probabilities = predict_emotion(
    image_path=str(test_image_path),
    model=model,
    transform=inference_transform,
    device=device
)

print(f"\nPrediction : {prediction}")
print(f"Confidence : {confidence:.2f}%\n")

print("Class Probabilities:")
for emotion, prob in probabilities.items():
    print(f"{emotion:<10}: {prob:.2f}%")

Testing with image: artifacts\data_ingestion\Organized\train\sad\train_00001_aligned.jpg

Prediction : happy
Confidence : 60.31%

Class Probabilities:
angry     : 4.25%
disgust   : 25.29%
fear      : 0.00%
happy     : 60.31%
neutral   : 0.00%
sad       : 10.14%
surprise  : 0.00%


### Batch inference

In [51]:
def batch_predict(folder_path, model, transform, device):
    """
    Predict emotions for all images in a folder.

    Args:
        folder_path (str): Folder containing images.
        model: Trained PyTorch model.
        transform: Inference transform.
        device: cpu or cuda.

    Returns:
        List of prediction results.
    """

    results = []

    supported_extensions = (".jpg", ".jpeg", ".png", ".bmp") 

    for image_name in os.listdir(folder_path):

        if not image_name.lower().endswith(supported_extensions):
            continue

        image_path = os.path.join(folder_path, image_name)

        prediction, confidence, probabilities = predict_emotion(
            image_path=image_path,
            model=model,
            transform=transform,
            device=device
        )

        results.append({
            "image": image_name,
            "prediction": prediction,
            "confidence": confidence,
            "probabilities": probabilities
        })

    return results

In [52]:
test_folder = Path('test_images')

results = batch_predict(
    folder_path=str(test_folder),
    model=model,
    transform=inference_transform,
    device=device
)

In [53]:
for result in results:

    print("-" * 50)

    print(f"Image      : {result['image']}")
    print(f"Prediction : {result['prediction']}")
    print(f"Confidence : {result['confidence']:.2f}%")

    print("\nClass Probabilities:")

    for emotion, prob in result["probabilities"].items():
        print(f"{emotion:<10}: {prob:.2f}%")

--------------------------------------------------
Image      : happy_2.jpg
Prediction : happy
Confidence : 44.70%

Class Probabilities:
angry     : 40.07%
disgust   : 1.48%
fear      : 0.00%
happy     : 44.70%
neutral   : 0.00%
sad       : 5.11%
surprise  : 8.63%
--------------------------------------------------
Image      : sad_1.jpg
Prediction : happy
Confidence : 92.48%

Class Probabilities:
angry     : 0.01%
disgust   : 0.49%
fear      : 0.00%
happy     : 92.48%
neutral   : 0.02%
sad       : 4.14%
surprise  : 2.86%


In [47]:
# Path relative to project root
test_sad_folder = Path("artifacts/data_ingestion/Organized/test/sad")
test_image_path = next(test_sad_folder.glob("*.jpg"))

prediction, confidence, probabilities = predict_emotion(
    image_path=str(test_image_path),
    model=model,
    transform=inference_transform,
    device=device
)

### Real time inference

In [30]:
# Load pretrained face detector
face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

In [49]:
# Replace webcam loop with face detector
cap = cv2.VideoCapture(0)

face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_detector.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(100, 100)
    )

    for (x, y, w, h) in faces:

        face = frame[y:y+h, x:x+w]

        prediction, confidence, _ = predict_emotion(
            model=model,
            transform=inference_transform,
            device=device,
            frame=face
        )

        cv2.rectangle(
            frame,
            (x, y),
            (x + w, y + h),
            (0, 255, 0),
            2
        )

        label = f"{prediction} ({confidence:.1f}%)"

        cv2.putText(
            frame,
            label,
            (x, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

    cv2.imshow("Emotion Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()